# 7 张量运算函数

学习目标
- 掌握张量相关运算函数

## 1. 常见运算函数

PyTorch 为每个张量封装了很多实用的计算函数，例如计算均值、平方根、求和等等。

In [20]:
import torch


def test():

    data = torch.randint(0, 10, [2, 3], dtype=torch.float64)
    print(data)
    print('-' * 50)

    # # 1. 计算均值
    # 注意: tensor 必须为 Float 或者 Double 类型
    print(data.mean())
    print(data.mean(dim=0))  # 按列计算均值
    print(data.mean(dim=1))  # 按行计算均值
    print('-' * 50)

    # 2. 计算总和
    print(data.sum())
    print(data.sum(dim=0))
    print(data.sum(dim=1))
    print('-' * 50)

    # 3. 计算平方
    print(data.pow(2))
    print('-' * 50)

    # 4. 计算平方根
    print(data.sqrt())
    print('-' * 50)

    # 5. 指数计算, e^n 次方
    print(data.exp())
    print('-' * 50)

    # 6. 对数计算
    print(data.log())  # 以 e 为底
    print(data.log2())
    print(data.log10())

if __name__ == '__main__':
    test()

tensor([[2., 0., 8.],
        [6., 3., 8.]], dtype=torch.float64)
--------------------------------------------------
tensor(4.5000, dtype=torch.float64)
tensor([4.0000, 1.5000, 8.0000], dtype=torch.float64)
tensor([3.3333, 5.6667], dtype=torch.float64)
--------------------------------------------------
tensor(27., dtype=torch.float64)
tensor([ 8.,  3., 16.], dtype=torch.float64)
tensor([10., 17.], dtype=torch.float64)
--------------------------------------------------
tensor([[ 4.,  0., 64.],
        [36.,  9., 64.]], dtype=torch.float64)
--------------------------------------------------
tensor([[1.4142, 0.0000, 2.8284],
        [2.4495, 1.7321, 2.8284]], dtype=torch.float64)
--------------------------------------------------
tensor([[7.3891e+00, 1.0000e+00, 2.9810e+03],
        [4.0343e+02, 2.0086e+01, 2.9810e+03]], dtype=torch.float64)
--------------------------------------------------
tensor([[0.6931,   -inf, 2.0794],
        [1.7918, 1.0986, 2.0794]], dtype=torch.float64)
tensor([

# 8 自动微分模块

学习目标
- 掌握梯度计算

自动微分（Autograd）模块对张量做了进一步的封装，具有自动求导功能。自动微分模块是构成神经网络训练的必要模块，在神经网络的反向传播过程中，Autograd 模块基于正向计算的结果对当前的参数进行微分计算，从而实现网络权重参数的更新。

## 1. 梯度基本计算

我们使用 `backward` 方法、`grad` 属性来实现梯度的计算和访问。

In [ ]:
import torch

# 1. 单标量梯度的计算
# y = x**2 + 20
def test01():

    # 定义需要求导的张量
    # 张量的值类型必须是浮点类型
    x = torch.tensor(10, requires_grad=True, dtype=torch.float64)
    # 变量经过中间运算
    f = x ** 2 + 20
    # 自动微分
    f.backward()
    # 打印 x 变量的梯度
    # backward 函数计算的梯度值会存储在张量的 grad 变量中
    print(x.grad)


# 2. 单向量梯度的计算
# y = x**2 + 20
def test02():

    # 定义需要求导的张量
    x = torch.tensor([10, 20, 30, 40], requires_grad=True, dtype=torch.float64)
    # 变量经过中间计算
    f1 = x ** 2 + 20

    # 注意：
    # 由于求导的结果必须是标量
    # 而 f 的结果是: tensor([120., 420.])
    # 所以，不能直接自动微分
    # 需要将结果计算为标量才能进行计算
    f2 = f1.mean()  # f2 = 1/2 * x

    # 自动微分
    f2.backward()

    # 打印 x 变量的梯度
    print(x.grad)


if __name__ == '__main__':
    test01()

## 2. 控制梯度计算

我们可以通过一些方法使得在 `requires_grad=True` 的张量在某些时候计算不进行梯度计算。

In [14]:
import torch

# 1. 控制不计算梯度
def test01():

    x = torch.tensor(10, requires_grad=True, dtype=torch.float64)
    print(x.requires_grad)

    # 第一种方式：对代码进行装饰
    with torch.no_grad():
        y = x ** 2
        print(y.requires_grad)

    # 第二种方式：对函数进行装饰
    @torch.no_grad()
    def my_func(x):
        return x ** 2

    print(my_func(x).requires_grad)

    # 第三种方式
    torch.set_grad_enabled(False)
    y = x ** 2
    print(y.requires_grad)

    # 恢复梯度计算，否则会影响后续代码
    torch.set_grad_enabled(True)
    
# 2. 注意: 累计梯度
def test02():

    # 定义需要求导张量
    x = torch.tensor([10, 20, 30, 40], requires_grad=True, dtype=torch.float64)

    for _ in range(3):

        f1 = x ** 2 + 20
        f2 = f1.mean()

        # 默认张量的 grad 属性会累计历史梯度值
        # 所以，需要我们每次手动清理上次的梯度
        # 注意: 一开始梯度不存在，需要做判断
        if x.grad is not None:
            x.grad.data.zero_()

        f2.backward()
        print(x.grad)
        
# # 3. 梯度下降优化最优解
def test03():

    # y = x**2
    x = torch.tensor(10, requires_grad=True, dtype=torch.float64)

    for _ in range(5000):

        # 正向计算
        f = x ** 2

        # 梯度清零
        if x.grad is not None:
            x.grad.data.zero_()

        # 反向传播计算梯度
        f.backward()

        # 更新参数
        x.data = x.data - 0.001 * x.grad

    print('%.10f' % x.data)


if __name__ == '__main__':
    test01()
    print('--------------------')
    test02()
    print('--------------------')
    test03()

True
False
False
False
--------------------
tensor([ 5., 10., 15., 20.], dtype=torch.float64)
tensor([ 5., 10., 15., 20.], dtype=torch.float64)
tensor([ 5., 10., 15., 20.], dtype=torch.float64)
--------------------
0.0004494759
